# EP2 — Agente Funcional: Asistente RRHH Invermar

> **Instrucciones:** Ejecutar las celdas **en orden** desde arriba hacia abajo.  
> Kernel requerido: **chatbot-invermar**  
> Si el kernel no aparece: `Kernel → Change Kernel → chatbot-invermar`

## ✅ Celda 1 — Verificación del entorno
> Corre esta celda primero. Debes ver 5 marcas verdes antes de continuar.

In [ ]:
import sys
print(f"Python: {sys.version}")

checks = [
    ("langchain_openai",   "ChatOpenAI"),
    ("langchain_community.vectorstores", "FAISS"),
    ("langchain_classic.agents", "AgentExecutor"),
    ("faiss",              None),
    ("dotenv",             None),
]

all_ok = True
for mod, cls in checks:
    try:
        m = __import__(mod, fromlist=[cls] if cls else [])
        print(f"  ✅ {mod}")
    except ImportError as e:
        print(f"  ❌ {mod} — {e}")
        all_ok = False

print()
if all_ok:
    print("🟢 Entorno listo. Continúa con la siguiente celda.")
else:
    print("🔴 Faltan paquetes. Asegúrate de usar el kernel 'chatbot-invermar'")

## ⚙️ Celda 2 — Imports y configuración

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from IPython.display import display, Markdown

print("📦 Imports OK")

# Busca el .env en el directorio actual y en los padres
env_loaded = False
for env_path in [Path.cwd() / ".env", *[p / ".env" for p in Path.cwd().parents]]:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        print(f"🔑 .env cargado desde: {env_path}")
        env_loaded = True
        break

if not env_loaded:
    print("❌ .env no encontrado. Verifica la ruta.")

TOKEN    = os.environ.get("GITHUB_TOKEN")
BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://models.inference.ai.azure.com")

print(f"🔐 GITHUB_TOKEN: {'OK (' + TOKEN[:20] + '...)' if TOKEN else '❌ NO ENCONTRADO'}")
print(f"🌐 BASE_URL    : {BASE_URL}")
print(f"📊 LangSmith   : {os.environ.get('LANGSMITH_TRACING', 'no configurado')}")

# Modelos
llm = ChatOpenAI(
    base_url=BASE_URL, api_key=TOKEN,
    model="gpt-4o-mini", temperature=0
)
embeddings = OpenAIEmbeddings(
    base_url=BASE_URL, api_key=TOKEN,
    model="text-embedding-3-small"
)

# Prueba rápida de conexión al LLM
test = llm.invoke([HumanMessage(content="Responde solo la palabra: OK")])
print(f"🤖 Prueba LLM  : {test.content.strip()}")
print()
print("✅ Celda 2 completada.")

## 📚 Celda 3 — Base de conocimiento (memoria largo plazo)
> FAISS indexa las políticas de Invermar para búsqueda semántica.

In [ ]:
politicas = [
    Document(
        page_content="Aguinaldo Fiestas Patrias: $85.000 líquidos para trabajadores con más de 3 meses de antigüedad. Más $15.000 por cada carga familiar acreditada.",
        metadata={"fuente": "Manual de Beneficios 2024"}
    ),
    Document(
        page_content="Aguinaldo de Navidad: $90.000 líquidos para trabajadores con contrato vigente al 15 de diciembre. Contratos a plazo fijo con más de 6 meses reciben $45.000.",
        metadata={"fuente": "Manual de Beneficios 2024"}
    ),
    Document(
        page_content="Feriado Progresivo (Art. 68 CT): 1 día adicional de vacaciones por cada 3 años trabajados, a partir de 10 años de imposiciones en cualquier sistema previsional.",
        metadata={"fuente": "Políticas de Vacaciones 2024"}
    ),
    Document(
        page_content="Procedimiento de Finiquito: La empresa dispone de 10 días hábiles para tener el finiquito disponible en Notaría de Puerto Montt o Calbuco, según la planta del trabajador.",
        metadata={"fuente": "Protocolo de Desvinculación 2024"}
    ),
    Document(
        page_content="Bono de Producción (Planta Procesos): Se calcula en base al volumen de materia prima procesada y se paga con un mes de desfase, en la remuneración del mes siguiente.",
        metadata={"fuente": "Anexo Remuneraciones Planta 2024"}
    ),
    Document(
        page_content="CCAF Los Andes: Invermar afilia a todos sus trabajadores de planta. Incluye préstamos sociales con tasa preferencial, subsidio dental y subsidio de fallecimiento.",
        metadata={"fuente": "Manual de Beneficios 2024"}
    ),
    Document(
        page_content="Fecha de Pago de Remuneraciones: El último día hábil de cada mes para todas las empresas del grupo Invermar. Si el último día es feriado, el pago se realiza el día hábil anterior.",
        metadata={"fuente": "Política de Pagos 2024"}
    ),
    Document(
        page_content="Licencias Médicas: Entregar en RRHH dentro de los 2 días hábiles de emisión. Invermar cubre el diferencial entre subsidio de Isapre/Fonasa y sueldo real durante los primeros 3 días.",
        metadata={"fuente": "Manual de Procedimientos RRHH 2024"}
    ),
]

print("⏳ Creando índice FAISS (puede tardar unos segundos)...")
vectorstore = FAISS.from_documents(politicas, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 2})

# Prueba del retriever
docs_prueba = retriever.invoke("aguinaldo fiestas patrias")
print(f"\n🔍 Prueba retriever: '{docs_prueba[0].page_content[:60]}...'")
print(f"\n✅ Base de conocimiento lista: {len(politicas)} políticas indexadas.")

## 🔧 Celda 4 — Herramientas del agente (5 tools)
> Incluye la herramienta `enviar_resumen_correo` que envía un correo al trabajador al final de cada consulta.
>
> **⚙️ Configurar correo:** Añade estas líneas a tu archivo `.env`:
> ```
> GMAIL_SENDER=tu_cuenta@gmail.com
> GMAIL_APP_PASSWORD=xxxx xxxx xxxx xxxx
> ```
> Obtén la App Password en: **Google Account → Seguridad → Contraseñas de aplicación**

In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText


@tool
def consultar_politicas_invermar(consulta: str) -> str:
    """Busca en la base de conocimiento oficial de Invermar S.A. información sobre
    beneficios, aguinaldos, vacaciones, finiquitos, bonos, CCAF, fechas de pago
    y políticas internas de RRHH. Usar cuando el trabajador pregunte sobre reglas,
    montos, procedimientos o beneficios internos de la empresa."""
    docs = retriever.invoke(consulta)
    if not docs:
        return "No se encontró información en las políticas de Invermar para esta consulta."
    return "\n\n".join(
        f"[{doc.metadata.get('fuente', 'Política interna')}]\n{doc.page_content}"
        for doc in docs
    )


@tool
def calcular_indemnizacion(anios_servicio: int, remuneracion_mensual: int) -> str:
    """Calcula la indemnización por años de servicio según el Art. 163 del Código del
    Trabajo chileno. Requiere: anios_servicio (años completos trabajados) y
    remuneracion_mensual (pesos chilenos, Art. 172 CT: sueldo base + promedio variables
    de los últimos 3 meses). Incluye tope legal de 11 años y aviso previo (Art. 162)."""
    tope         = min(anios_servicio, 11)
    indem_anios  = tope * remuneracion_mensual
    aviso_previo = remuneracion_mensual
    total        = indem_anios + aviso_previo
    return (
        f"CÁLCULO DE INDEMNIZACIÓN (Art. 163 CT)\n"
        f"{'─'*40}\n"
        f"  Años trabajados  : {anios_servicio} años\n"
        f"  Años con tope    : {tope} (máx. legal: 11)\n"
        f"  Remuneración base: ${remuneracion_mensual:,.0f}\n"
        f"{'─'*40}\n"
        f"  Indemn. años     : ${indem_anios:,.0f}  ({tope} × ${remuneracion_mensual:,.0f})\n"
        f"  Aviso previo     : ${aviso_previo:,.0f}  (Art. 162 CT)\n"
        f"{'─'*40}\n"
        f"  TOTAL ESTIMADO   : ${total:,.0f}\n"
        f"  ⚠️  No incluye vacaciones proporcionales."
    )


@tool
def calcular_vacaciones_proporcionales(meses_trabajados: int, remuneracion_mensual: int) -> str:
    """Calcula los días y monto de vacaciones proporcionales según el Art. 73 del Código
    del Trabajo chileno. Requiere: meses_trabajados en el año en curso (1-12) y
    remuneracion_mensual en pesos chilenos. Usar cuando el trabajador quiera saber
    cuánto le corresponde por vacaciones no tomadas al momento del finiquito."""
    meses_trabajados = max(1, min(12, meses_trabajados))
    dias_anuales     = 15
    dias_prop        = round((dias_anuales * meses_trabajados) / 12, 1)
    rem_diaria       = remuneracion_mensual / 30
    monto            = round(dias_prop * rem_diaria)
    return (
        f"VACACIONES PROPORCIONALES (Art. 73 CT)\n"
        f"{'─'*40}\n"
        f"  Meses en el año  : {meses_trabajados}\n"
        f"  Días hábiles/año : {dias_anuales}\n"
        f"  Días proporc.    : {dias_prop} días hábiles\n"
        f"  Rem. diaria      : ${rem_diaria:,.0f}\n"
        f"{'─'*40}\n"
        f"  MONTO A PAGAR    : ${monto:,.0f}\n"
    )


@tool
def clasificar_consulta(consulta: str) -> str:
    """Clasifica el tipo de consulta laboral para orientar al trabajador al área
    correcta de la empresa. Categorías: LIQUIDACION, VACACIONES, BENEFICIOS,
    FINIQUITO, NORMATIVA, OTRO. Usar cuando la consulta sea ambigua o cuando
    necesites determinar qué área de la empresa debe atender al trabajador."""
    c = consulta.lower()
    if any(w in c for w in ["finiquito", "despido", "desvinculaci", "indemnizaci"]):
        return "FINIQUITO — Derivar a Remuneraciones para cálculo y firma en notaría."
    if any(w in c for w in ["vacacion", "vacación", "feriado", "días libres"]):
        return "VACACIONES — Aplica Art. 67-68 CT (legales y progresivas)."
    if any(w in c for w in ["sueldo", "liquidacion", "liquidación", "descuento", "haber", "remuner", "pago"]):
        return "LIQUIDACION — Revisar con el área de Remuneraciones."
    if any(w in c for w in ["beneficio", "aguinaldo", "bono", "ccaf", "caja", "préstamo"]):
        return "BENEFICIOS — Consultar Manual de Beneficios en portal BUK."
    if any(w in c for w in ["código", "codigo", "artículo", "articulo", "ley", "norma", "contrato"]):
        return "NORMATIVA — Consultar Código del Trabajo y normativa interna."
    return "OTRO — Derivar al área de Recursos Humanos para orientación."


@tool
def enviar_resumen_correo(resumen: str) -> str:
    """Envía un resumen de la consulta al trabajador por correo electrónico.
    Llamar SIEMPRE al FINAL de cada interacción con un texto claro que incluya
    la información clave entregada en la respuesta (montos, fechas, políticas,
    normativa consultada). El destinatario es rosoriolleucun@gmail.com."""
    destinatario = "rosoriolleucun@gmail.com"
    remitente    = os.environ.get("GMAIL_SENDER", "")
    password     = os.environ.get("GMAIL_APP_PASSWORD", "")

    if not remitente or not password:
        return (
            "⚠️ Credenciales de correo no configuradas. "
            "Añade GMAIL_SENDER y GMAIL_APP_PASSWORD al archivo .env. "
            "El resumen NO fue enviado."
        )

    resumen_esc = (resumen
                   .replace("&", "&amp;")
                   .replace("<", "&lt;")
                   .replace(">", "&gt;"))

    html_body = f"""<html><body>
<div style="font-family:Arial,sans-serif;max-width:600px;margin:0 auto;">
  <div style="background:#00205B;padding:20px 24px;border-radius:8px 8px 0 0;">
    <h2 style="color:#C9B777;margin:0;font-size:18px;">🐟 Agente RRHH — Invermar S.A.</h2>
    <p style="color:#fff;margin:4px 0 0;font-size:12px;">Resumen de consulta laboral</p>
  </div>
  <div style="background:#fff;padding:24px;border:1px solid #e0e0e0;
              border-top:none;border-radius:0 0 8px 8px;">
    <p style="color:#333;white-space:pre-wrap;line-height:1.7;font-size:14px;">{resumen_esc}</p>
    <hr style="border:none;border-top:1px solid #eee;margin:20px 0;">
    <p style="color:#aaa;font-size:11px;margin:0;">
      Generado automáticamente por el Agente RRHH de Invermar S.A.<br>
      Para consultas adicionales contacte al área de Remuneraciones.
    </p>
  </div>
</div>
</body></html>"""

    try:
        msg            = MIMEMultipart("alternative")
        msg["Subject"] = "📋 Resumen consulta RRHH — Invermar S.A."
        msg["From"]    = remitente
        msg["To"]      = destinatario
        msg.attach(MIMEText(html_body, "html"))

        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(remitente, password)
            server.sendmail(remitente, destinatario, msg.as_string())

        return f"✅ Resumen enviado por correo a {destinatario}"
    except Exception as e:
        return f"❌ Error al enviar correo: {e}"


tools = [
    consultar_politicas_invermar,
    calcular_indemnizacion,
    calcular_vacaciones_proporcionales,
    clasificar_consulta,
    enviar_resumen_correo,
]

print(f"✅ {len(tools)} herramientas registradas:")
for t in tools:
    print(f"   🔧 {t.name}")

## 🧠 Celda 5 — Memoria conversacional + Agente ReAct

In [ ]:
# ── Memoria de corto plazo ────────────────────────────────────────────────────
MEMORY_WINDOW = 6

class WindowChatMessageHistory(InMemoryChatMessageHistory):
    """Historial con ventana deslizante para controlar el tamaño del contexto."""
    max_messages: int = MEMORY_WINDOW

    def add_message(self, message: BaseMessage) -> None:
        super().add_message(message)
        if len(self.messages) > self.max_messages:
            self.messages = self.messages[-self.max_messages:]

store: dict = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = WindowChatMessageHistory()
    return store[session_id]

# ── Prompt del sistema con few-shot ──────────────────────────────────────────
SYSTEM_PROMPT = """\
Eres el Agente de Recursos Humanos de Invermar S.A., especializado en remuneraciones,
beneficios y normativa laboral chilena. Cubres las empresas: Invermar S.A.,
Pesquera La Portada S.A., La Península S.A. y Astilleros Calbuco S.A.

HERRAMIENTAS DISPONIBLES:
• consultar_politicas_invermar        → busca en manuales y políticas internas.
• calcular_indemnizacion              → cálculo de indemnización Art. 163 CT.
• calcular_vacaciones_proporcionales  → cálculo de vacaciones Art. 73 CT.
• clasificar_consulta                 → categoriza consultas ambiguas.
• enviar_resumen_correo               → envía resumen por email al trabajador.

SECUENCIA OBLIGATORIA — en CADA respuesta sigue este orden exacto:
  1. Llama a las herramientas necesarias para responder.
  2. Redacta tu respuesta final al trabajador.
  3. Llama a enviar_resumen_correo → es el ÚLTIMO paso, siempre, sin excepción.

════════════════════════════════════════════
EJEMPLOS DEL PATRÓN CORRECTO:
════════════════════════════════════════════

Ejemplo 1 — consulta de beneficio:
  Entrada: "¿Cuánto es el aguinaldo de Fiestas Patrias?"
  Acción 1: consultar_politicas_invermar("aguinaldo fiestas patrias")
  Respuesta: [explico el monto y condiciones]
  Acción 2: enviar_resumen_correo("Aguinaldo Fiestas Patrias: $85.000 líquidos para trabajadores con más de 3 meses. Más $15.000 por carga familiar acreditada. Fuente: Manual de Beneficios 2024.")

Ejemplo 2 — cálculo de indemnización:
  Entrada: "Llevo 7 años, gano $700.000, ¿cuánto me corresponde de indemnización?"
  Acción 1: calcular_indemnizacion(7, 700000)
  Respuesta: [explico el desglose del cálculo]
  Acción 2: enviar_resumen_correo("Indemnización Art. 163 CT: 7 años × $700.000 = $4.900.000. Aviso previo Art. 162: $700.000. Total estimado: $5.600.000. No incluye vacaciones proporcionales.")

Ejemplo 3 — finiquito completo (multi-herramienta):
  Entrada: "Llevo 6 años, me voy en agosto, gano $600.000. ¿Qué me corresponde?"
  Acción 1: clasificar_consulta("finiquito 6 años agosto $600.000")
  Acción 2: calcular_indemnizacion(6, 600000)
  Acción 3: calcular_vacaciones_proporcionales(8, 600000)
  Respuesta: [explico el desglose completo]
  Acción 4: enviar_resumen_correo("Finiquito estimado: Indemnización $3.600.000 + Aviso previo $600.000 + Vacaciones proporcionales (8 meses) $160.000. Total: $4.360.000.")

════════════════════════════════════════════

REGLAS ADICIONALES:
- Responde siempre en español, tono profesional y cercano.
- Usa Markdown (negritas, listas) para estructurar tus respuestas.
- No inventes cifras; usa siempre las herramientas para datos concretos.
- Si la consulta está fuera de tu alcance, deriva al área de Remuneraciones.
- NUNCA finalices una respuesta sin haber llamado a enviar_resumen_correo.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# ── Agente ReAct ──────────────────────────────────────────────────────────────
agent          = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False,
    max_iterations=8,
    handle_parsing_errors=True,
)

print("✅ Agente ReAct listo.")
print(f"   Memoria conversacional : ventana de {MEMORY_WINDOW} mensajes")
print(f"   Herramientas           : {len(tools)}")
print(f"   Max iteraciones/turno  : 8")

## 💬 Celda 6 — Interfaz visual + función `chat()`
> Ejecuta esta celda para ver el panel de chat interactivo. También define `chat()` para uso programático en los ejemplos de abajo.

In [ ]:
import ipywidgets as widgets
import mistune
from IPython.display import display, HTML

SESSION_ID = "invermar_ep2"

# ─── Renderizado de burbujas ──────────────────────────────────────────────────
def _burbuja_usuario(texto: str) -> HTML:
    esc = texto.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    return HTML(f"""<div style="display:flex;justify-content:flex-end;margin:6px 4px;">
  <div style="background:#00205B;color:#fff;padding:10px 14px;
              border-radius:18px 18px 4px 18px;max-width:78%;
              font-family:Arial,sans-serif;font-size:14px;line-height:1.5;">
    <b>🧑 Tú</b><br>{esc}
  </div>
</div>""")

def _burbuja_agente(texto: str) -> HTML:
    html_md = mistune.html(texto)
    return HTML(f"""<div style="display:flex;justify-content:flex-start;margin:6px 4px;">
  <div style="background:#fff;border-left:4px solid #C9B777;padding:10px 14px;
              border-radius:0 18px 18px 0;max-width:82%;
              font-family:Arial,sans-serif;font-size:14px;line-height:1.6;
              box-shadow:0 1px 4px rgba(0,0,0,.08);">
    <b style="color:#00205B;">🤖 Agente Invermar</b><br>{html_md}
  </div>
</div>""")

def _bienvenida_html() -> str:
    return """<div style="text-align:center;padding:40px 20px;">
  <div style="font-size:44px;margin-bottom:12px;">🐟</div>
  <p style="font-size:15px;margin:0;color:#555;font-family:Arial,sans-serif;">
    Bienvenido/a al <b>Agente de RRHH de Invermar S.A.</b>
  </p>
  <p style="font-size:12px;margin:10px 0 0;color:#aaa;font-family:Arial,sans-serif;">
    Pregúntame sobre beneficios, remuneraciones, vacaciones, finiquitos y normativa laboral.
  </p>
</div>"""

# ─── Construcción de la interfaz ──────────────────────────────────────────────
header_w = widgets.HTML("""
<div style="background:#00205B;padding:16px 20px;border-radius:10px 10px 0 0;">
  <h2 style="color:#C9B777;margin:0;font-family:Arial,sans-serif;font-size:18px;">
    🐟 Agente RRHH — Invermar S.A.
  </h2>
  <p style="color:#fff;margin:4px 0 0;font-size:12px;opacity:.8;">
    Consultas de remuneraciones, beneficios y normativa laboral chilena
  </p>
</div>""")

chat_out = widgets.Output(layout=widgets.Layout(
    min_height="420px", max_height="540px",
    overflow_y="auto", border="1px solid #e0e0e0",
    padding="12px 8px", background_color="#f7f7f7",
))

text_in = widgets.Text(
    placeholder="Escribe tu consulta aquí...",
    layout=widgets.Layout(flex="1", height="40px"),
)
send_btn = widgets.Button(
    description="▶ Enviar",
    layout=widgets.Layout(width="110px", height="40px"),
)
send_btn.style.button_color = "#00205B"

clear_btn = widgets.Button(
    description="🔄 Nueva sesión",
    button_style="warning",
    layout=widgets.Layout(width="145px", height="40px"),
)

status_w = widgets.HTML(
    '<span style="font-size:11px;color:#888">✅ Agente listo — escribe tu consulta</span>'
)

input_row = widgets.HBox(
    [text_in, send_btn, clear_btn],
    layout=widgets.Layout(
        gap="6px", padding="8px",
        background_color="#efefef",
        border="1px solid #e0e0e0", border_top="none",
    ),
)
status_row = widgets.HBox(
    [status_w],
    layout=widgets.Layout(
        padding="4px 12px", background_color="#fafafa",
        border="1px solid #e0e0e0", border_top="none",
        border_radius="0 0 10px 10px",
    ),
)
ui = widgets.VBox(
    [header_w, chat_out, input_row, status_row],
    layout=widgets.Layout(
        max_width="800px", border="1px solid #ccc",
        border_radius="10px", overflow="hidden",
        box_shadow="0 2px 12px rgba(0,0,0,.12)",
    ),
)

# ─── Lógica de eventos ────────────────────────────────────────────────────────
def _invocar(msg: str) -> str:
    historial = get_session_history(SESSION_ID)
    resp      = agent_executor.invoke({
        "input"       : msg,
        "chat_history": historial.messages,
    })
    output = resp["output"]
    historial.add_message(HumanMessage(content=msg))
    historial.add_message(AIMessage(content=output))
    return output

def on_send(b=None):
    msg = text_in.value.strip()
    if not msg:
        return
    text_in.value     = ""
    send_btn.disabled = True
    status_w.value    = '<span style="font-size:11px;color:#C9B777">⏳ Procesando...</span>'

    with chat_out:
        display(_burbuja_usuario(msg))

    try:
        output = _invocar(msg)
        with chat_out:
            display(_burbuja_agente(output))
        status_w.value = '<span style="font-size:11px;color:#28a745">✅ Respuesta lista</span>'
    except Exception as e:
        with chat_out:
            display(HTML(f'<div style="color:red;padding:10px;font-family:Arial">❌ Error: {e}</div>'))
        status_w.value = '<span style="font-size:11px;color:red">❌ Error — revisa la consola</span>'

    send_btn.disabled = False

def on_clear(b=None):
    store.pop(SESSION_ID, None)
    chat_out.clear_output()
    status_w.value = '<span style="font-size:11px;color:#888">🔄 Nueva sesión iniciada</span>'
    with chat_out:
        display(HTML(_bienvenida_html()))

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
text_in.on_submit(on_send)

# ─── Función programática (usada por las celdas de ejemplos) ──────────────────
def chat(mensaje: str, session_id: str = SESSION_ID) -> str:
    """Envía un mensaje al agente desde código y muestra el resultado en el widget."""
    _hist = get_session_history(session_id)
    with chat_out:
        display(_burbuja_usuario(mensaje))
    print(f"⏳ {mensaje[:60]}...")
    try:
        resp   = agent_executor.invoke({"input": mensaje, "chat_history": _hist.messages})
        output = resp["output"]
        _hist.add_message(HumanMessage(content=mensaje))
        _hist.add_message(AIMessage(content=output))
        with chat_out:
            display(_burbuja_agente(output))
        return output
    except Exception as e:
        print(f"❌ Error: {e}")
        return ""

def nueva_sesion(session_id: str = SESSION_ID) -> None:
    """Reinicia la conversación (borra el historial)."""
    store.pop(session_id, None)
    chat_out.clear_output()
    with chat_out:
        display(HTML(_bienvenida_html()))
    print("🔄 Sesión reiniciada.")

# ─── Mostrar interfaz ─────────────────────────────────────────────────────────
with chat_out:
    display(HTML(_bienvenida_html()))

print("💬 Interfaz lista — el panel interactivo aparece abajo:")
display(ui)

---
## 🧪 Ejemplos de uso

A partir de aquí puedes correr cada celda de forma independiente. Cada una demuestra una capacidad distinta del agente.

### Ejemplo 1 — Consulta de política interna
El agente usa `consultar_politicas_invermar` para buscar en FAISS.

In [ ]:
nueva_sesion()
chat("¿Cuánto es el aguinaldo de Fiestas Patrias y quién lo recibe?")

### Ejemplo 2 — Cálculo de indemnización
El agente usa `calcular_indemnizacion` con los datos que entrega el trabajador.

In [ ]:
nueva_sesion()
chat(
    "Llevo 9 años en Pesquera La Portada y me van a despedir por necesidades de la empresa. "
    "Mi sueldo base es $680.000 y en los últimos 3 meses tuve un promedio de bonos de $120.000. "
    "¿Cuánto me corresponde de indemnización?"
)

### Ejemplo 3 — Finiquito completo (multi-herramienta)
El agente encadena `clasificar_consulta` → `calcular_indemnizacion` → `calcular_vacaciones_proporcionales`.

In [ ]:
nueva_sesion()
chat(
    "Soy operadora en Astilleros Calbuco, llevo 6 años. Me voy a ir en agosto (mes 8). "
    "Mi sueldo es $600.000 sin variables. ¿Qué me corresponde en el finiquito?"
)

### Ejemplo 4 — Memoria conversacional (3 preguntas de seguimiento)
El agente recuerda el contexto sin que el trabajador repita información.

In [ ]:
nueva_sesion()
chat("¿Cuándo me pagan el sueldo en Invermar?")

In [ ]:
chat("¿Y si ese día es feriado?")  # sin nueva_sesion → recuerda el contexto

In [ ]:
chat("¿Vale para todas las empresas del grupo?")

### Ejemplo 5 — Consulta fuera del dominio
El agente detecta que no puede responder y deriva al área correcta.

In [ ]:
nueva_sesion()
chat("Tengo un problema con mi jefe, ¿puedo pedir cambio de turno sin que me afecte el sueldo?")

---
## 💡 Escribe tu propia consulta

```python
nueva_sesion()          # opcional: reinicia el historial
chat("tu pregunta aquí")
```

In [ ]:
nueva_sesion()
chat("¿Qué beneficios tiene la CCAF Los Andes?")

---
## 🔍 Inspección de la memoria conversacional

In [ ]:
historial = get_session_history(SESSION_ID)
msgs = historial.messages
print(f"📊 Mensajes en memoria: {len(msgs)} / {MEMORY_WINDOW} (ventana máxima)\n")

for i, msg in enumerate(msgs, 1):
    tipo    = "🧑 Usuario" if isinstance(msg, HumanMessage) else "🤖 Agente"
    preview = (msg.content[:90] + "...") if len(msg.content) > 90 else msg.content
    print(f"[{i}] {tipo}: {preview}")